# 03 · Évaluer et décider : du modèle à la campagne

**Séance 5 · deux blocs d'environ 25 minutes en séance, 30 minutes en autonomie.**

Question directrice, seconde moitié : **« à qui devrions-nous envoyer la prochaine campagne, et qu'est-ce que cela rapporte ? »**

En séance 4, nous avons construit un arbre de décision et mesuré son exactitude (accuracy) : 85,7 %, à peine mieux que le modèle idiot. Nous avions conclu que l'exactitude (accuracy) ne suffit pas à juger un modèle. Aujourd'hui, **même arbre, mêmes prédictions** : seule la manière de les évaluer change. On regarde quelles erreurs il fait, ce que chacune coûte, et on choisit un **seuil** de décision en unités monétaires.

**Bloc 1 — Quelles erreurs, à quel coût ?** matrice de confusion, précision (precision) et rappel (recall), trois stratégies comparées, résultat en unités monétaires.
**Bloc 2 — Décider :** le seuil, le point mort, le prix de l'honnêteté chronologique, et ce qu'un modèle ne prouve pas.

## Bloc 1 — Quelles erreurs, à quel coût ?

### 1. Retrouver l'arbre de la séance 4

Même première cellule, même préparation, même séparation, même fonction d'entraînement : l'arbre est **exactement** celui de la séance 4.

In [ ]:
# Première cellule de chaque notebook : récupérer le module du cours, puis charger le fichier brut.
import os
import urllib.request

URL_MODULE = "https://raw.githubusercontent.com/2501Pr0ject/Data4Sup2Pub/main/notebooks/preparation.py"
if not os.path.exists("preparation.py"):
    urllib.request.urlretrieve(URL_MODULE, "preparation.py")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import preparation as prep

brut = prep.charger()

In [ ]:
df = prep.preparer(brut, verbose=False)            # mêmes règles qu'en séance 4, sans réimprimer le journal
train, test = prep.separer_train_test(df)          # même séparation 70 / 30, même graine

cible, variables = prep.CIBLE, prep.JEU_COMPLET
X_train, y_train = train[variables], train[cible]
X_test, y_test = test[variables], test[cible]

arbre = prep.entrainer_arbre(X_train, y_train)     # profondeur par défaut : 3, comme en séance 4
print(f"Arbre : {arbre.get_n_leaves()} feuilles | exactitude (accuracy) sur le test : {arbre.score(X_test, y_test):.1%}")
print(f"Jeu de test : {len(test)} clients, dont {int(y_test.sum())} répondants")

### 2. Deux façons de se tromper

**Question marketing :** *quand le modèle se trompe, de quelle façon se trompe-t-il ?*

**La notion.** Pour chaque client du jeu de test, le modèle annonce « répondra » ou « ne répondra pas », et la réalité est connue. Quatre cas possibles, rangés dans la **matrice de confusion** :

| | Réalité : a répondu | Réalité : n'a pas répondu |
|---|---|---|
| **Annoncé « répondra »** | vrai positif (VP) : contacté, a acheté | faux positif (FP) : contacté pour rien |
| **Annoncé « ne répondra pas »** | faux négatif (FN) : non contacté, aurait acheté | vrai négatif (VN) : non contacté, à raison |

Les deux erreurs n'ont pas le même coût : un faux positif coûte un contact ; un faux négatif coûte une vente manquée.

In [ ]:
from sklearn.metrics import confusion_matrix

def bilan(prediction):
    # Compte les quatre cas pour une prédiction (0/1) sur le jeu de test
    vn, fp, fn, vp = confusion_matrix(y_test, prediction, labels=[0, 1]).ravel()
    return {"VP": int(vp), "FP": int(fp), "FN": int(fn), "VN": int(vn)}

def matrice(b):
    # Présente le bilan sous forme de tableau 2 × 2
    return pd.DataFrame([[b["VP"], b["FP"]], [b["FN"], b["VN"]]],
                        index=["annoncé « répondra »", "annoncé « ne répondra pas »"],
                        columns=["a répondu", "n'a pas répondu"])

prediction_arbre = arbre.predict(X_test)       # 1 si l'arbre annonce « répondra », 0 sinon
bilan_arbre = bilan(prediction_arbre)
matrice(bilan_arbre)

**Question 1.** Lisez la matrice : combien de clients l'arbre fait-il contacter, combien parmi eux achètent, combien d'acheteurs manque-t-il ? Laquelle des deux erreurs vous paraît la plus grave pour l'enseigne, et de quoi cela dépend-il ?

### 3. Trois mesures au lieu d'une

**La notion.** De la matrice on tire trois mesures, à toujours nommer avec leur terme anglais :

- **exactitude (accuracy)** = (VP + VN) / tous : part de clients bien classés ;
- **précision (precision)** = VP / (VP + FP) : parmi les clients contactés, part de ceux qui achètent ;
- **rappel (recall)** = VP / (VP + FN) : parmi les acheteurs, part de ceux qu'on a contactés.

Précision (precision) et rappel (recall) tirent dans des sens opposés : contacter plus de monde augmente le rappel (recall) et baisse la précision (precision).

In [ ]:
def mesures(b):
    # Les trois mesures à partir du bilan ; la précision (precision) n'existe pas si personne n'est contacté
    contactes = b["VP"] + b["FP"]
    return {"contactés": contactes,
            "exactitude (accuracy)": (b["VP"] + b["VN"]) / sum(b.values()),
            "précision (precision)": b["VP"] / contactes if contactes else float("nan"),
            "rappel (recall)": b["VP"] / (b["VP"] + b["FN"])}

pd.Series(mesures(bilan_arbre)).round(3)

### 4. Trois stratégies côte à côte

**Question marketing :** *l'arbre vaut-il mieux que ce que l'équipe marketing ferait sans modèle ?*

**La notion.** Un modèle ne se juge pas seul : il se compare à ce qu'on ferait sans lui. Trois références :

- le **modèle idiot** : ne contacter personne (ou tout le monde) ;
- la **règle métier** : « contacter les clients qui ont déjà accepté au moins une campagne » (`Campagnes_acceptees ≥ 1`), ce qu'une équipe appliquerait à la main ;
- l'**arbre** de la séance 4.

In [ ]:
strategies = {
    "personne (modèle idiot)": np.zeros(len(y_test), dtype=int),
    "tout le monde": np.ones(len(y_test), dtype=int),
    "règle métier : Campagnes_acceptees ≥ 1": (X_test["Campagnes_acceptees"] >= 1).astype(int).to_numpy(),
    "arbre (séance 4)": prediction_arbre,
}
bilans = {nom: bilan(pred) for nom, pred in strategies.items()}
comparaison = pd.DataFrame({nom: {**b, **mesures(b)} for nom, b in bilans.items()}).T
comparaison = comparaison.astype({c: int for c in ["VP", "FP", "FN", "VN", "contactés"]})
comparaison.round(3)

**Question 2.** Classez les quatre stratégies selon l'exactitude (accuracy), puis selon le nombre d'acheteurs trouvés (VP). Les deux classements sont-ils les mêmes ? Que vaut la règle métier face à l'arbre ?

### 5. Le résultat en unités monétaires

**Question marketing :** *combien rapporte chaque stratégie ?*

**La notion.** Le fichier contenait deux constantes, retirées à la préparation mais conservées dans le module : **un contact coûte 3**, **un client qui accepte rapporte 11** (unités monétaires du cas d'origine, « MU »). Le résultat d'une stratégie est donc :

résultat = 11 × VP − 3 × (VP + FP)

Chaque faux positif coûte 3 ; chaque faux négatif est une vente à 11 − 3 = 8 de marge qu'on n'a pas faite, mais elle n'apparaît pas dans le résultat : c'est un manque à gagner, pas une dépense.

In [ ]:
COUT, REVENU = prep.COUT_CONTACT, prep.REVENU_REPONDANT
print(f"Coût d'un contact : {COUT} | revenu d'un client qui accepte : {REVENU}")

def resultat(b, cout=COUT, revenu=REVENU):
    return revenu * b["VP"] - cout * (b["VP"] + b["FP"])

comparaison["résultat (MU)"] = [resultat(b) for b in bilans.values()]
# Une stratégie n'est utilisable que si tout ce qu'elle demande était connu au moment de décider (notebook 01, section 9)
comparaison["utilisable au moment de la décision ?"] = ["oui", "oui", "oui", "non établi"]
comparaison[["contactés", "VP", "FP", "FN", "résultat (MU)", "utilisable au moment de la décision ?"]]

**Question 3.** Quelle stratégie rapporte le plus ? Que rapporte « tout le monde » sur ces 608 clients, et à quoi cela ressemble-t-il dans l'histoire du fichier (notebook 00, question 4) ? L'arbre bat-il la règle métier, et pourquoi la dernière colonne tempère-t-elle ce verdict ?

## Bloc 2 — Décider

### 6. Du « oui / non » à la probabilité

**Question marketing :** *à partir de quelle chance de réponse vaut-il la peine de contacter un client ?*

**La notion.** L'arbre ne dit pas seulement « oui » ou « non » : pour chaque client, il donne la **part de répondants dans sa feuille**, qu'on lit comme une probabilité de réponse. « Annoncer répondra » voulait dire jusqu'ici « probabilité ≥ 0,5 ». Mais ce seuil de 0,5 n'a rien de naturel. Un contact coûte 3 et rapporte 11 s'il aboutit : un client vaut d'être contacté dès que 11 × p > 3, c'est-à-dire **p > 3 / 11 ≈ 27 %**. C'est le **point mort**, et il se calcule avant de regarder le modèle.

In [ ]:
proba = arbre.predict_proba(X_test)[:, 1]       # probabilité de réponse estimée pour chaque client du test
point_mort = prep.POINT_MORT
print(f"Point mort = {COUT} / {REVENU} = {point_mort:.3f} : on contacte si la probabilité estimée dépasse {point_mort:.1%}")

# Un arbre de 8 feuilles ne connaît que 8 probabilités : une par feuille
feuilles = pd.DataFrame({"probabilité estimée": proba.round(3), "a répondu": y_test.values})
feuilles = feuilles.groupby("probabilité estimée").agg(clients=("a répondu", "size"), répondants=("a répondu", "sum"))
feuilles["taux réel"] = (feuilles["répondants"] / feuilles["clients"]).round(3)
feuilles["à contacter ?"] = np.where(feuilles.index > point_mort, "oui", "non")
feuilles

**Question 4.** Combien de valeurs différentes la probabilité peut-elle prendre, et pourquoi ? Quelles feuilles passent le point mort ? Comparez, pour chaque feuille, la probabilité estimée (apprise sur l'apprentissage) et le taux réel (observé sur le test) : les feuilles « tiennent-elles » sur des clients jamais vus ?

### 7. Le curseur de seuil

**Formulaire.** Déplacez le curseur, relancez la cellule, lisez le résultat. Essayez 0,25, 0,5, 0,8, puis cherchez le seuil qui rapporte le plus. La cellule qui précède le curseur rappelle les seules valeurs que la probabilité peut prendre : **le résultat ne change qu'en franchissant l'une d'elles**.

In [ ]:
valeurs_possibles = sorted(feuilles.index)
print("Probabilités possibles, une par feuille :", " | ".join(f"{v:.1%}" for v in valeurs_possibles))
print("Entre deux de ces valeurs, déplacer le curseur ne change aucune décision.")

In [ ]:
#@title Seuil de décision
seuil = 0.5  #@param {type:"slider", min:0.05, max:0.95, step:0.05}

prediction_seuil = (proba >= seuil).astype(int)      # « répondra » si la probabilité estimée atteint le seuil
b = bilan(prediction_seuil)
m = mesures(b)
print(f"Seuil {seuil:.2f} : {m['contactés']} clients contactés | VP {b['VP']}, FP {b['FP']}, FN {b['FN']}")
print(f"   précision (precision) {m['précision (precision)']:.1%} | rappel (recall) {m['rappel (recall)']:.1%} | "
      f"exactitude (accuracy) {m['exactitude (accuracy)']:.1%}")
print(f"   résultat : {resultat(b)} MU")
matrice(b)

**Question 5.** Essayez 0,25, 0,50 et 0,80. Que se passe-t-il à 0,15, et à 0,90 ? Quel seuil rapporte le plus, et à combien de contacts ?

### 8. Le résultat selon le seuil

Plutôt que de tâtonner, on calcule le résultat pour tous les seuils et on le trace. Les points marquent les huit probabilités des feuilles ; la ligne verticale rouge est le point mort calculé à la section 6, **sans regarder le modèle**.

In [ ]:
seuils = np.round(np.arange(0.05, 0.96, 0.05), 2)
courbe = pd.DataFrame([{"seuil": s, **mesures(bilan((proba >= s).astype(int))),
                        "résultat (MU)": resultat(bilan((proba >= s).astype(int)))} for s in seuils]).set_index("seuil")

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.step(courbe.index, courbe["résultat (MU)"], where="post", color="#3b6ea5", linewidth=2, label="résultat selon le seuil")
resultats_feuilles = [resultat(bilan((proba >= v).astype(int))) for v in valeurs_possibles]
ax.scatter(valeurs_possibles, resultats_feuilles, color="#3b6ea5", zorder=3, s=36, label="probabilités des feuilles")
for v in valeurs_possibles:
    ax.axvline(v, color="#bfbfbf", linestyle=":", linewidth=1)
ax.axvline(point_mort, color="#c0392b", linestyle="--", linewidth=1.5, label=f"point mort {point_mort:.2f}")
ax.axhline(0, color="#7f7f7f", linewidth=1)
ax.set_xlabel("seuil de décision")
ax.set_ylabel("résultat (MU) sur les 608 clients du test")
ax.set_title("Le résultat avance par marches : une marche à chaque probabilité de feuille", loc="left")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#e5e5e5")
ax.set_axisbelow(True)
plt.show()

meilleur = courbe["résultat (MU)"].idxmax()
print(f"Meilleur résultat : {courbe.loc[meilleur, 'résultat (MU)']} MU, atteint dès le seuil {meilleur:.2f} "
      f"({int(courbe.loc[meilleur, 'contactés'])} contacts)")

**Question 6.** Pourquoi le résultat avance-t-il par marches plutôt que de varier continûment ? Combien de marches compte-t-on, et que faudrait-il pour que la courbe soit plus lisse ?

**Question 7.** Le point mort (27 %) et le seuil qui rapporte le plus coïncident-ils ? Est-ce une coïncidence ? Fallait-il le jeu de test pour trouver le seuil ?

### 9. Le prix de l'honnêteté chronologique

**Question marketing :** *que vaut le modèle si l'on ne garde que les variables dont on est sûr qu'elles étaient connues avant la campagne ?*

**La notion.** En séance 4, nous avons gardé toutes les variables pour l'exercice, en sachant que les dépenses et la récence n'ont pas de chronologie établie (notebook 01, section 9). On entraîne ici un second arbre, **même profondeur, même séparation, même graine**, sur les seules **variables sûres** : profil et campagnes passées. Les deux modèles sont évalués sur les mêmes 608 clients, au même seuil.

In [ ]:
variables_sures = prep.JEU_SUR
arbre_sur = prep.entrainer_arbre(train[variables_sures], y_train)
proba_sur = arbre_sur.predict_proba(test[variables_sures])[:, 1]

lignes = {}
for nom, p in [("jeu complet (27 variables)", proba), ("variables sûres (12 variables)", proba_sur)]:
    for s, etiquette in [(0.5, "seuil 0,5"), (point_mort, "seuil = point mort")]:
        b = bilan((p >= s).astype(int))
        lignes[(nom, etiquette)] = {**mesures(b), "résultat (MU)": resultat(b)}
cote_a_cote = pd.DataFrame(lignes).T.astype({"contactés": int, "résultat (MU)": int})
cote_a_cote.index.names = ["modèle", "seuil"]
cote_a_cote.round(3)

In [ ]:
# Bilan final, toutes stratégies au seuil qui leur convient, avec la question qui compte : utilisable ou non ?
bilan_final = pd.DataFrame({
    "personne (modèle idiot)": {**bilans["personne (modèle idiot)"], "utilisable ?": "oui"},
    "tout le monde": {**bilans["tout le monde"], "utilisable ?": "oui"},
    "règle métier : Campagnes_acceptees ≥ 1": {**bilans["règle métier : Campagnes_acceptees ≥ 1"], "utilisable ?": "oui"},
    "arbre variables sûres, point mort": {**bilan((proba_sur >= point_mort).astype(int)), "utilisable ?": "oui"},
    "arbre jeu complet, point mort": {**bilan((proba >= point_mort).astype(int)), "utilisable ?": "non établi"},
}).T
bilan_final.insert(0, "contactés", bilan_final["VP"] + bilan_final["FP"])
bilan_final["résultat (MU)"] = [resultat(bilan_final.loc[i, ["VP", "FP", "FN", "VN"]].to_dict()) for i in bilan_final.index]
bilan_final[["contactés", "VP", "FP", "résultat (MU)", "utilisable ?"]]

In [ ]:
importances_sur = pd.Series(arbre_sur.feature_importances_, index=variables_sures)
print("Variables utilisées par l'arbre « variables sûres » :")
print(importances_sur[importances_sur > 0].sort_values(ascending=False).round(2).to_string())

**Question 8.** Au point mort, combien rapporte chaque arbre ? Que coûte le passage aux variables sûres, et que gagne-t-on en échange ?

**Question 9.** Regardez le bilan final à information égale, c'est-à-dire entre stratégies utilisables : la règle métier (+128) bat l'arbre sur variables sûres (+63). Comment un modèle peut-il perdre contre une règle d'une ligne ? Quelle recommandation présentez-vous à la direction, et à quelle condition l'arbre complet deviendrait-il défendable ?

### 10. Corrélation, prédiction, effet : trois choses différentes

Pas de code ici. L'arbre a appris que les clients ayant accepté des campagnes passées répondent davantage. Trois phrases, trois statuts :

1. « Les clients ayant accepté des campagnes passées répondent plus souvent. » — une **corrélation**, observée dans le fichier.
2. « Si l'on contacte les clients ayant accepté des campagnes passées, environ 37 % d'entre eux achèteront. » — une **prédiction**, valable si l'on contacte comme lors du pilote (même offre, même canal, même période).
3. « Envoyer plus de campagnes rend les clients plus réactifs. » — un **effet** supposé d'une intervention : rien dans le fichier ne permet de l'affirmer.

**Question 10.** Un directeur lit le +195 MU et conclut : « le modèle a prouvé que la campagne est rentable ». Qu'est-ce qui est juste et qu'est-ce qui est faux dans cette phrase ? Que faudrait-il faire pour prouver l'efficacité d'une campagne ciblée ?

### 11. Synthèse métier

- Le même arbre qu'en séance 4, évalué autrement : il fait contacter 58 clients et en convertit 32 ; il manque 61 acheteurs. L'exactitude (accuracy) ne voyait rien de tout cela.
- En unités monétaires : contacter tout le monde perd 801, la règle métier gagne 128, l'arbre gagne 178 au seuil par défaut et **195 au point mort**.
- Le **seuil se calcule à partir des coûts** (3 / 11 ≈ 27 %), pas en tâtonnant sur le modèle. Le modèle sert à estimer les probabilités, et il n'en connaît que huit : le résultat avance par marches.
- À information égale, **la règle métier (+128) bat l'arbre sur variables sûres (+63)** : c'est elle, la recommandation présentable. L'arbre complet (+195) n'est défendable que si une date d'extraction prouve la chronologie de la récence et des dépenses.
- Corrélation, prédiction, effet : le modèle reconnaît, il ne prouve pas. Seul un test contrôlé prouve l'efficacité d'une campagne.

### 12. Limites

- Les résultats en MU portent sur 608 clients du test, avec une seule séparation ; l'unité monétaire du fichier n'est pas documentée.
- Un arbre de 8 feuilles ne connaît que 8 probabilités : le curseur agit par paliers. Un modèle plus fin donnerait une courbe plus lisse, pas forcément un meilleur résultat.
- La courbe de la section 8 est calculée sur le jeu de test : lire son maximum pour choisir le seuil serait optimiser sur le test. Ici, le seuil vient des coûts.
- Tout repose sur une campagne pilote envoyée au hasard à 2 240 clients ; les coûts et revenus sont ceux du cas d'origine.

### En autonomie (30 minutes)

La direction annonce que le coût d'un contact passe de 3 à **5** (envoi postal au lieu d'un appel), le revenu d'une vente restant 11.

1. Calculez le nouveau point mort à la main. (5 min)
2. Réglez le curseur de la section 7 sur le seuil juste au-dessus de ce point mort (pas de 0,05) et notez : contacts, VP, FP, résultat **recalculé avec le coût 5** (la cellule utilise le coût 3 : refaites le calcul 11 × VP − 5 × contacts à la main). Comparez avec le coût 3. (15 min)
3. Écrivez trois phrases pour la direction : ce que change la hausse du coût de contact, ce que le modèle permet, ce qu'il ne prouve pas. (10 min)